Post-calculate Autocorrelation RMSE for model checkpoints and then log them to tensorboard

Imports

In [ ]:
import os
import re
from pathlib import Path
from collections import OrderedDict
import torch
from torch.utils.tensorboard import SummaryWriter
from src.data_utils import unpickle_data
from src.modeling import HolsteinPARC
from src.analysis_utils import gen_preds_like, compare_cdw_order_vis, compare_site_vis, compare_cdw_order_autocorrelation, PLTArgs, calc_charge_order, calc_lattice_order, calc_autocorrelation_traj
from src.utils import get_project_root
from src.training_utils import np_RMSE

Arguments

In [9]:
checkpoint_dir = "/scratch/rnx2bc/research/holstein-parc/checkpoints/L_32-quench_0_1-steps_1000-training/medium_parc_model-deep-fast_train-1a/20251018-232218"

max_lag: int | float | None = None

max_batch_size: int = 256 # Maximum number of trajectories that can be inferenced at once

Prepare All Materials

In [ ]:
checkpoint_dir = Path(checkpoint_dir)

config = unpickle_data(checkpoint_dir / "LAST_CONFIG.pkl")

## Get data

data_name = get_project_root() / "data" / "generated" / config.data_names[0] / "pickled" / "val.pkl"
if config.data_nicknames is not None:
    data_nickname = config.data_nicknames[0]
else:
    if len(config.data_names) > 1:
        data_nickname = config.data_names[0]
    else:
        data_nickname = ""


data = unpickle_data(data_name)

## Resolve max lag

data_n_steps = data[0].shape[1] - 1

if max_lag is None:
    max_lag = 0.25

if isinstance(max_lag, float):
    if max_lag > 1:
        raise ValueError(f"max_lag must be less than 1 if float, got {max_lag}")
    max_lag = int(data_n_steps * max_lag)
    print(f"Setting max lag at {max_lag}/{data_n_steps} steps")


## Resolve model paths
re_pattern = r'epoch_(\d+)\.pth'
model_paths = OrderedDict()
for path in checkpoint_dir.iterdir():
    match = re.fullmatch(re_pattern, path.name)
    if match:
        model_paths[int(match.group(1))] = path.name

model_paths = OrderedDict(sorted(model_paths.items(), key=lambda x: x[0]))

if not model_paths:
    raise ValueError(f"No model checkpoints found in directory with pattern '{re_pattern}'")


print("Epochs found:")
for model_path in model_paths.values():
    print("\t", model_path)

## Get tensorboard writer
logdir = Path(checkpoint_dir.as_posix().replace("/checkpoints/", "/runs/"))
if not logdir.is_dir():
    raise ValueError(f"Logdir {logdir} does not exist!")
writer = SummaryWriter(logdir)

## Misc

def load_model(model_path):
    model_kwargs = unpickle_data(os.path.join(os.path.dirname(model_path), "model_kwargs.pkl"))
    model = HolsteinPARC(**model_kwargs)
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu'), weights_only=True))
    return model


Setting max lag at 250/1000 steps
Epochs found:
	 epoch_1.pth
	 epoch_2.pth
	 epoch_3.pth
	 epoch_4.pth
	 epoch_5.pth
	 epoch_6.pth
	 epoch_7.pth
	 epoch_8.pth
	 epoch_9.pth
	 epoch_10.pth
	 epoch_11.pth
	 epoch_12.pth
	 epoch_13.pth
	 epoch_14.pth
	 epoch_15.pth
	 epoch_16.pth
	 epoch_17.pth
	 epoch_18.pth
	 epoch_19.pth
	 epoch_20.pth
	 epoch_21.pth
	 epoch_22.pth
	 epoch_23.pth
	 epoch_24.pth
	 epoch_25.pth
	 epoch_26.pth
	 epoch_27.pth
	 epoch_28.pth
	 epoch_29.pth


Post-Calculate Autocorr for Each Model

In [ ]:
for i, (epoch_num, model_path) in enumerate(model_paths.items()):
    ## Load model
    model_path = checkpoint_dir / model_path
    model = load_model(model_path)

    rho_label, Q_label, _ = data
    ## Model inference
    rho_pred, Q_pred, _ = gen_preds_like(model, labels=data, max_batch_size=max_batch_size, suppress_output=True)

    # Calculate CDW orders
    label_charge_order = calc_charge_order(rho_label)
    pred_charge_order = calc_charge_order(rho_pred)

    label_lattice_order = calc_lattice_order(Q_label)
    pred_lattice_order = calc_lattice_order(Q_pred)

    ## Calculate Lags
    label_charge_order_autocorr_traj = calc_autocorrelation_traj(label_charge_order, max_lag=max_lag)
    pred_charge_order_autocorr_traj = calc_autocorrelation_traj(pred_charge_order, max_lag=max_lag)

    label_lattice_order_autocorr_traj = calc_autocorrelation_traj(label_lattice_order, max_lag=max_lag)
    pred_lattice_order_autocorr_traj = calc_autocorrelation_traj(pred_lattice_order, max_lag=max_lag)

    ## Get combined RMSE of autocorr trajs
    charge_order_autocorr_rmse = np_RMSE(label_charge_order_autocorr_traj, pred_charge_order_autocorr_traj)
    lattice_order_autocorr_rmse = np_RMSE(label_lattice_order_autocorr_traj, pred_lattice_order_autocorr_traj)
    total_autocorr_rmse = charge_order_autocorr_rmse + lattice_order_autocorr_rmse
    total_autocorr_rmse = total_autocorr_rmse.item()

    ## log to tensorboard
    writer.add_scalar(f"val/{data_nickname}/autocorr_rmse_{max_lag}", total_autocorr_rmse, epoch_num)
    print(f"Completed epoch {epoch_num}: {(i/len(model_paths)) * 100:.0f}% done!")


writer.close()


Completed epoch 1: 0% done!
Completed epoch 2: 3% done!
Completed epoch 3: 7% done!
Completed epoch 4: 10% done!
Completed epoch 5: 14% done!
Completed epoch 6: 17% done!
Completed epoch 7: 21% done!
Completed epoch 8: 24% done!
Completed epoch 9: 28% done!
Completed epoch 10: 31% done!
Completed epoch 11: 34% done!
Completed epoch 12: 38% done!
Completed epoch 13: 41% done!
Completed epoch 14: 45% done!
Completed epoch 15: 48% done!
Completed epoch 16: 52% done!
Completed epoch 17: 55% done!
Completed epoch 18: 59% done!
Completed epoch 19: 62% done!
Completed epoch 20: 66% done!
Completed epoch 21: 69% done!
Completed epoch 22: 72% done!
Completed epoch 23: 76% done!
Completed epoch 24: 79% done!
Completed epoch 25: 83% done!
Completed epoch 26: 86% done!
Completed epoch 27: 90% done!
Completed epoch 28: 93% done!
Completed epoch 29: 97% done!
